# SDSS AITS Multi-Epoch Downloader

This downloader is designed to run simultaneously across multiple instances using a shared Drive folder, utilising `NUM_SHARDS = N` and unique `SHARD_ID` (0…N-1) per node. Machines automatically coordinate through periodic directory rescans and stop together when every class hits `LIMIT_PER_CLASS`.

Key features of the downloader are:

- **Dual-Crop Generation**: Single network call produces two datasets (`original/` for original centred crop and `augmented/` spatial offset crop for augmentation purposes).
- **Temporal Metadata**: Preserves observation timestamps (`mjd`) alongside `images`, `filter`, and `label` in serialised `.pkl` files.
- **Quality Filtering**: Enforces thresholds for `MIN_MJDS`, `MIN_BANDS`, and `MIN_BASELINE_DAYS` with criteria-based rejection logging.
- **Classes Count**:  Four classes (**SNIa**, **pSNIa**, **AGN**, **Variable**) will be downloaded into the collective drive.

In [ ]:
!pip install astroquery

In [ ]:
import os
import gc
import glob
import time
import random
import pickle
import tempfile
import warnings
import threading
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed, wait, FIRST_COMPLETED

import numpy as np
from astropy.table import Table
from astropy import coordinates as coords
import astropy.units as u
from astropy.wcs import WCS
from astropy.nddata import Cutout2D
from astroquery.sdss import SDSS

warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Configuration

**To split work across accounts, change only `NUM_SHARDS` and `SHARD_ID`:**

| Account | `NUM_SHARDS` | `SHARD_ID` |
|---|---|---|
| 1 | N | 0 |
| 2 | N | 1 |
| … | N | … |
| N | N | N−1 |

Everything else (especially `BASE_DIR` and `LIMIT_PER_CLASS`) must be **identical** on every account.

In [ ]:
# ===== SHARDING CONFIG =====
NUM_SHARDS = 20     # total active instances running concurrently
SHARD_ID = 0        # instance index (0 to NUM_SHARDS-1)

# ===== FLEET-WIDE TARGET =====
LIMIT_PER_CLASS = 300  # target quota per class across all instances

# ===== PATHS =====
BASE_DIR = "/content/drive/MyDrive/sdss_v2/"
CATALOG_PATH = "/content/drive/MyDrive/sdss/master_data.txt"

# ===== CLASSES & BANDS =====
TARGET_CLASSES = ["pSNIa"]
BAND_MAP = {"u": 1, "g": 2, "r": 3, "i": 4, "z": 5}
BANDS = ["u", "g", "r", "i", "z"]

# ===== CROPPING =====
CROP_SIZE = 32
JITTER_PX = 4  # fixed pixel offset magnitude for spatial augmentation

# ===== QUALITY FILTERS =====
MIN_MJDS = 15            # minimum required observation epochs
MIN_BANDS = 3            # minimum required photometric bands
MIN_BASELINE_DAYS = 14   # minimum temporal coverage (mjd_max - mjd_min)

# ===== CONCURRENCY =====
OBJECT_WORKERS = 16
DOWNLOAD_WORKERS = 10
CROP_WORKERS = 8

# ===== RETRY & HOUSEKEEPING =====
MAX_RETRIES = 3
BACKOFF_BASE = 2.0              # exponential backoff base factor (seconds)
JITTER_MAX = 0.5                # maximum random delay prior to HTTP request
TMP_SWEEP_INTERVAL = 120        # interval for temporary file cleanup (seconds)
TMP_SWEEP_MIN_AGE = 1200        # minimum age threshold for stale temporary files (seconds)
GLOBAL_REFRESH_INTERVAL = 90    # sync interval for shared directory state (seconds)

# ===== SHARED STATE =====
global_lock = threading.Lock()
global_disk_counts = {c: 0 for c in TARGET_CLASSES}
local_inflight = {c: 0 for c in TARGET_CLASSES}
locally_saved_since_scan = {c: 0 for c in TARGET_CLASSES}
rejection_counts = defaultdict(lambda: defaultdict(int))
print_lock = threading.Lock()
stop_event = threading.Event()

print(f"Shard {SHARD_ID}/{NUM_SHARDS} | classes {TARGET_CLASSES}")
print(
    f"Filters: >={MIN_MJDS} epochs, >={MIN_BANDS} bands,"
    f" >={MIN_BASELINE_DAYS}d baseline"
)
print(f"Output: {BASE_DIR}original/  and  {BASE_DIR}augmented/")

Shard 0/20 | classes ['pSNIa']
Filters: >=15 epochs, >=3 bands, >=14d baseline
Output: /content/drive/MyDrive/sdss_v2/original/  and  /content/drive/MyDrive/sdss_v2/augmented/


# Helpers and objects count tracking

In [ ]:
def safe_print(msg):
    with print_lock:
        print(msg, flush=True)


def with_retry(fn, desc):
    for attempt in range(MAX_RETRIES):
        try:
            time.sleep(random.uniform(0, JITTER_MAX))
            return fn()
        except Exception as e:
            wait_s = BACKOFF_BASE * (2**attempt) + random.uniform(0, 1)
            safe_print(
                f"  ! {desc} failed (attempt {attempt+1}/{MAX_RETRIES}): {e} "
                f"-> retrying in {wait_s:.1f}s"
            )
            time.sleep(wait_s)
    safe_print(f"  ! {desc} failed after {MAX_RETRIES} attempts, giving up.")
    return None


def scan_disk_counts():
    counts = {}
    for obj_class in TARGET_CLASSES:
        save_dir = os.path.join(BASE_DIR, "original", obj_class)
        if not os.path.isdir(save_dir):
            counts[obj_class] = 0
            continue
        counts[obj_class] = len(
            [f for f in os.listdir(save_dir) if f.endswith(".pickle")]
        )
    return counts


def effective_progress(obj_class):
    with global_lock:
        return (
            global_disk_counts[obj_class]
            + local_inflight[obj_class]
            + locally_saved_since_scan[obj_class]
        )


def reserve_slot(obj_class):
    with global_lock:
        effective = (
            global_disk_counts[obj_class]
            + local_inflight[obj_class]
            + locally_saved_since_scan[obj_class]
        )
        if effective >= LIMIT_PER_CLASS:
            return False
        local_inflight[obj_class] += 1
        return True


def release_slot(obj_class):
    with global_lock:
        local_inflight[obj_class] = max(0, local_inflight[obj_class] - 1)


def mark_saved(obj_class):
    with global_lock:
        local_inflight[obj_class] = max(0, local_inflight[obj_class] - 1)
        locally_saved_since_scan[obj_class] += 1


def all_classes_full():
    with global_lock:
        return all(
            global_disk_counts[c]
            + local_inflight[c]
            + locally_saved_since_scan[c]
            >= LIMIT_PER_CLASS
            for c in TARGET_CLASSES
        )


def refresh_global_counts_loop():
    while not stop_event.is_set():
        time.sleep(GLOBAL_REFRESH_INTERVAL)
        counts = scan_disk_counts()
        with global_lock:
            for cls in TARGET_CLASSES:
                global_disk_counts[cls] = counts[cls]
                locally_saved_since_scan[cls] = 0
        if all_classes_full():
            stop_event.set()


def sweep_stale_tmp_files():
    tmp_dir = tempfile.gettempdir()
    while not stop_event.is_set():
        time.sleep(TMP_SWEEP_INTERVAL)
        now = time.time()
        removed, freed = 0, 0
        for path in glob.glob(os.path.join(tmp_dir, "tmp*")):
            try:
                if not os.path.isfile(path):
                    continue
                if now - os.path.getmtime(path) < TMP_SWEEP_MIN_AGE:
                    continue
                freed += os.path.getsize(path)
                os.remove(path)
                removed += 1
            except OSError:
                pass
        if removed:
            safe_print(
                f"  [tmp-sweep] removed {removed} stale files, freed"
                f" {freed/1e6:.1f} MB"
            )

# Fetch and dual-crop

In [ ]:
def object_jitter_offset(cid):
    """
    Generates a deterministic per-object pixel offset based on Candidate ID (CID)
    Rerolls (0, 0) to ensure the augmented copy always differs from the original
    """
    rng = random.Random(int(cid))
    while True:
        dx = rng.randint(-JITTER_PX, JITTER_PX)
        dy = rng.randint(-JITTER_PX, JITTER_PX)
        if (dx, dy) != (0, 0):
            return dx, dy


def fetch_query_region(pos, cid):
    """Queries multi-epoch observation metadata for a target coordinate using astroquery SDSS"""
    return with_retry(
        lambda: SDSS.query_region(
            pos, radius=2 * u.arcsec, data_release=17, cache=False
        ),
        desc=f"query_region CID {cid}",
    )


def fetch_band_images(single_match, band, cid, match_idx):
    """Retrieves FITS image frames from SDSS for a given observation match and filter band"""
    return with_retry(
        lambda: SDSS.get_images(
            matches=single_match,
            band=band,
            data_release=17,
            show_progress=False,
            cache=False,
        ),
        desc=f"get_images CID {cid} band {band} epoch {match_idx}",
    )


def _finite_cutout(data, x, y):
    """Extracts a CROP_SIZE x CROP_SIZE array at (x, y); returns None if out of bounds or contains non-finite values"""
    try:
        cut = Cutout2D(
            data,
            position=(float(x), float(y)),
            size=CROP_SIZE,
            mode="partial",
            fill_value=np.nan,
        )
    except Exception:
        return None
    arr = cut.data.astype(np.float32).copy()
    if arr.shape == (CROP_SIZE, CROP_SIZE) and np.all(np.isfinite(arr)):
        return arr
    return None


def crop_band_images(hdul_list, band, pos, jitter_dx, jitter_dy):
    """
    Transforms celestial coordinates via WCS and extracts dual cutouts (centered and spatial-shifted)
    Discards frames where either cutout is invalid or missing timestamp data
    """
    frames = []
    for hdul in hdul_list:
        try:
            header = hdul[0].header
            data = hdul[0].data

            mjd = header.get(
                "MJD", header.get("MJD-OBS", header.get("TAI", None))
            )
            if mjd is None:
                continue

            try:
                wcs = WCS(header)
                x, y = wcs.world_to_pixel(pos)
            except Exception:
                continue
            if not (np.isfinite(x) and np.isfinite(y)):
                continue

            img_center = _finite_cutout(data, x, y)
            img_jitter = _finite_cutout(data, x + jitter_dx, y + jitter_dy)
            if img_center is None or img_jitter is None:
                continue

            frames.append({
                "image_center": img_center,
                "image_jitter": img_jitter,
                "band": BAND_MAP[band],
                "mjd": float(mjd),
            })
        finally:
            hdul.close()
    return frames


def validate_frames(frames):
    """Evaluates collected frames against observational quality thresholds (epochs, bands, temporal baseline)"""
    if not frames:
        return False, "no_valid_frames", None
    mjds = np.array([f["mjd"] for f in frames])
    bands = sorted({f["band"] for f in frames})
    n_epochs = len(np.unique(np.round(mjds, 4)))
    baseline = float(mjds.max() - mjds.min())
    stats = {
        "n_epochs": n_epochs,
        "n_bands": len(bands),
        "bands": bands,
        "baseline_days": baseline,
    }
    if n_epochs < MIN_MJDS:
        return False, "low_mjds", stats
    if len(bands) < MIN_BANDS:
        return False, "low_bands", stats
    if baseline < MIN_BASELINE_DAYS:
        return False, "short_baseline", stats
    return True, "pass", stats

# Per-object coordinator

In [ ]:
def process_row(row, download_executor, crop_executor):
    """
    Process a single astronomical target row by handling the region query, concurrent multi-band frame download, 
    dual-crop extraction, quality validation, and atomic pickle serialisation
    """
    if stop_event.is_set():
        return

    obj_class = str(row["Classification"]).strip()
    if obj_class not in TARGET_CLASSES:
        return

    cid = row["CID"]
    ra = float(row["RA"])
    dec = float(row["DEC"])
    filename = f"{cid}_{ra:.5f}_{dec:.5f}.pickle"

    out_original = os.path.join(BASE_DIR, "original", obj_class, filename)
    out_augmented = os.path.join(BASE_DIR, "augmented", obj_class, filename)

    if os.path.exists(out_original) and os.path.exists(out_augmented):
        safe_print(f"[{obj_class}] Skipping CID {cid} - already downloaded.")
        return

    if not reserve_slot(obj_class):
        return
    if all_classes_full():
        stop_event.set()

    pos = coords.SkyCoord(ra=ra * u.deg, dec=dec * u.deg)
    jitter_dx, jitter_dy = object_jitter_offset(cid)

    xid = download_executor.submit(fetch_query_region, pos, cid).result()
    if xid is None or len(xid) == 0:
        safe_print(f"  -> No catalog match for CID {cid}, skipping.")
        rejection_counts[obj_class]["no_catalog_match"] += 1
        release_slot(obj_class)
        gc.collect()
        return

    num_obs = len(xid)
    total_jobs = num_obs * len(BANDS)
    safe_print(
        f"[{obj_class}: {effective_progress(obj_class)}/{LIMIT_PER_CLASS}"
        f" fleet-wide] CID {cid} - {num_obs} epochs x {len(BANDS)} bands (jitter"
        f" offset {jitter_dx:+d},{jitter_dy:+d})..."
    )

    valid_frames = []
    pending_download = {}
    for i in range(num_obs):
        single_match = xid[i : i + 1]
        for band in BANDS:
            fut = download_executor.submit(
                fetch_band_images, single_match, band, cid, i
            )
            pending_download[fut] = (i, band)

    pending_crop = set()
    download_done = 0
    crop_done = 0

    while pending_download or pending_crop:
        done, _ = wait(
            set(pending_download) | pending_crop, return_when=FIRST_COMPLETED
        )
        for fut in done:
            if fut in pending_download:
                i, band = pending_download.pop(fut)
                download_done += 1
                hdul_list = fut.result()
                if hdul_list:
                    crop_fut = crop_executor.submit(
                        crop_band_images,
                        hdul_list,
                        band,
                        pos,
                        jitter_dx,
                        jitter_dy,
                    )
                    pending_crop.add(crop_fut)
            else:
                pending_crop.discard(fut)
                crop_done += 1
                valid_frames.extend(fut.result())

        if crop_done % len(BANDS) == 0 or (
            not pending_download and not pending_crop
        ):
            safe_print(
                f"  [CID {cid}] downloaded {download_done}/{total_jobs}, cropped"
                f" {crop_done}/{total_jobs} ({len(valid_frames)} frames"
                " collected)"
            )

    passes, reason, stats = validate_frames(valid_frames)
    if not passes:
        safe_print(f"  -> CID {cid} REJECTED ({reason}): {stats}")
        rejection_counts[obj_class][reason] += 1
        release_slot(obj_class)
        gc.collect()
        return

    safe_print(
        f"  -> CID {cid} passed filters: {stats['n_epochs']} epochs,"
        f" {stats['n_bands']} bands, {stats['baseline_days']:.0f}d baseline"
    )

    valid_frames.sort(key=lambda x: x["mjd"])
    filters = np.array([f["band"] for f in valid_frames], dtype=int)
    mjds = np.array([f["mjd"] for f in valid_frames], dtype=np.float64)

    for dataset_type, img_key, out_file in (
        ("original", "image_center", out_original),
        ("augmented", "image_jitter", out_augmented),
    ):
        images = np.stack([f[img_key] for f in valid_frames])
        data_object = {
            "images": images,
            "filter": filters,
            "label": obj_class,
            "mjd": mjds,
        }
        os.makedirs(os.path.dirname(out_file), exist_ok=True)
        tmp_file = out_file + ".tmp"
        with open(tmp_file, "wb") as fh:
            pickle.dump(data_object, fh, protocol=pickle.HIGHEST_PROTOCOL)
        os.rename(tmp_file, out_file)

    mark_saved(obj_class)
    safe_print(
        f"  -> SUCCESS! Saved {len(valid_frames)} frames to original/ +"
        f" augmented/ [{obj_class}:"
        f" {effective_progress(obj_class)}/{LIMIT_PER_CLASS} fleet-wide]"
    )
    gc.collect()

# Main donwloader

In [ ]:
def main():
    print("Loading catalog...")
    catalog = Table.read(CATALOG_PATH, format="ascii.cds")
    os.makedirs(BASE_DIR, exist_ok=True)
    print(f"Catalog loaded: {len(catalog)} rows.")

    initial_counts = scan_disk_counts()
    with global_lock:
        for cls in TARGET_CLASSES:
            global_disk_counts[cls] = initial_counts[cls]
    for obj_class in TARGET_CLASSES:
        safe_print(f"  {obj_class}: {effective_progress(obj_class)}/{LIMIT_PER_CLASS} "
                   f"already on disk fleet-wide.")

    print(f"\nShard {SHARD_ID}/{NUM_SHARDS} starting. "
          f"Pools: {OBJECT_WORKERS} object / {DOWNLOAD_WORKERS} download / {CROP_WORKERS} crop.\n")

    threading.Thread(target=sweep_stale_tmp_files, daemon=True).start()
    threading.Thread(target=refresh_global_counts_loop, daemon=True).start()

    with ThreadPoolExecutor(max_workers=OBJECT_WORKERS) as object_executor, \
         ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as download_executor, \
         ThreadPoolExecutor(max_workers=CROP_WORKERS) as crop_executor:

        futures = []
        for idx, row in enumerate(catalog):
            if idx % NUM_SHARDS != SHARD_ID:
                continue
            if stop_event.is_set():
                break
            obj_class = str(row['Classification']).strip()
            if obj_class not in TARGET_CLASSES:
                continue
            if effective_progress(obj_class) >= LIMIT_PER_CLASS:
                continue
            futures.append(object_executor.submit(
                process_row, row, download_executor, crop_executor))

        for fut in as_completed(futures):
            exc = fut.exception()
            if exc is not None:
                safe_print(f"  ! Unhandled error in worker: {exc}")

    print("\nProcessing complete!")
    print(f"Final fleet-wide counts (original/): {scan_disk_counts()}")
    print("\nRejection summary (this shard only):")
    for obj_class in TARGET_CLASSES:
        if rejection_counts[obj_class]:
            print(f"  {obj_class}:")
            for reason, count in sorted(rejection_counts[obj_class].items(), key=lambda kv: -kv[1]):
                print(f"    - {reason}: {count}")

In [ ]:
main()

Loading catalog...
Catalog loaded: 10258 rows.
  pSNIa: 0/300 already on disk fleet-wide.

Shard 0/20 starting. Pools: 16 object / 10 download / 8 crop.

[pSNIa: 16/300 fleet-wide] CID 7762 - 12 epochs x 5 bands (jitter offset +0,+1)...
[pSNIa: 16/300 fleet-wide] CID 2342 - 35 epochs x 5 bands (jitter offset +4,-1)...
[pSNIa: 16/300 fleet-wide] CID 8808 - 21 epochs x 5 bands (jitter offset +3,+1)...
[pSNIa: 16/300 fleet-wide] CID 8549 - 12 epochs x 5 bands (jitter offset -2,-4)...
  -> No catalog match for CID 7712, skipping.
  -> No catalog match for CID 10620, skipping.
[pSNIa: 15/300 fleet-wide] CID 12310 - 2 epochs x 5 bands (jitter offset +1,+1)...
[pSNIa: 16/300 fleet-wide] CID 7073 - 34 epochs x 5 bands (jitter offset -1,+0)...
  -> No catalog match for CID 3506, skipping.
[pSNIa: 16/300 fleet-wide] CID 13429 - 8 epochs x 5 bands (jitter offset -3,+0)...
  [CID 7762] downloaded 1/60, cropped 0/60 (0 frames collected)
  -> No catalog match for CID 7357, skipping.
[pSNIa: 16/300 f